# Chapter 7.7: CUTLASS & CuTe for GEMM Optimization

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand the CUTLASS library** and its role in the GPU ecosystem
2. **Explain CuTe** (CUDA Tensor Expressions) layout abstractions
3. **Customize GEMM epilogues** for fused post-processing
4. **Work with tensor core operations** (WMMA, MMA)
5. **Understand CUTLASS 3.x** changes and CuTe integration
6. **Integrate CUTLASS kernels** into Python workflows
7. **Compare CUTLASS vs cuBLAS vs Triton** performance

## Prerequisites

- Chapter 7.1 (CUDA Primer)
- Chapter 7.2 (Triton Programming)
- Understanding of matrix multiplication tiling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.7_cutlass.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.7_cutlass.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. CUTLASS Overview

### 1.1 What is CUTLASS?

**CUTLASS** (CUDA Templates for Linear Algebra Subroutines) is NVIDIA's open-source library for high-performance GEMM (General Matrix Multiply) and related operations. It provides:

- **Templated C++ kernels** that can be customized for specific use cases
- **Near-cuBLAS performance** with much more flexibility
- **Epilogue customization** — fuse post-GEMM operations (bias, activation, quantization)
- **Support for all data types** — FP64, FP32, FP16, BF16, FP8, INT8, INT4

### 1.2 Why Not Just Use cuBLAS?

| Feature | cuBLAS | CUTLASS |
|---------|--------|--------|
| Performance | Best for standard GEMM | 95-100% of cuBLAS |
| Customization | None | Full epilogue, layout, precision |
| Epilogue fusion | No | Yes (bias, activation, quant) |
| Data types | Standard (FP16, FP32, INT8) | All including FP8, INT4, mixed |
| Source code | Closed | Open source |
| Integration | Link library | Header-only templates |

### 1.3 CUTLASS in LLM Serving

CUTLASS is used in vLLM/SGLang for:
- **FP8 GEMM** on H100 (not well-supported by cuBLAS initially)
- **Quantized GEMM** (INT8, INT4) with custom dequantization
- **Fused GEMM + activation** (e.g., GEMM + SiLU + multiply)
- **Mixed-precision GEMM** (FP16 inputs, FP32 accumulation, FP16 output)

## 2. CUTLASS Architecture

### 2.1 Hierarchical Decomposition

CUTLASS decomposes GEMM into a hierarchy that maps to GPU hardware:

```
Level 0: Device (Grid)       — Partition C into thread block tiles
Level 1: Thread Block        — Cooperative tile loading + computation
Level 2: Warp                — MMA (Matrix Multiply-Accumulate) operations
Level 3: Thread              — Register-level computation
```

### 2.2 GEMM Components

```
CUTLASS GEMM = Mainloop + Epilogue

Mainloop:
  - Global memory → Shared memory (tiled loading)
  - Shared memory → Registers (MMA feeding)
  - MMA computation (tensor cores)

Epilogue:
  - Post-processing the accumulator
  - Examples: bias add, activation, type conversion
  - Store results to global memory
```

### 2.3 Key Concepts

| Concept | Description |
|---------|------------|
| **ThreadblockShape** | Tile size computed by one thread block (e.g., 128×128×32) |
| **WarpShape** | Tile size computed by one warp (e.g., 64×64×32) |
| **InstructionShape** | Single MMA instruction size (e.g., 16×8×16 for FP16) |
| **Stages** | Pipeline depth for overlapping loads and computation |
| **Epilogue** | Post-GEMM operations (bias, activation, quantization) |

In [ ]:
# Visualize the CUTLASS GEMM decomposition
def visualize_cutlass_decomposition(
    M=4096, N=4096, K=4096,
    tb_m=128, tb_n=128, tb_k=32,
    warp_m=64, warp_n=64, warp_k=32,
    mma_m=16, mma_n=8, mma_k=16,
):
    """
    Show the hierarchical decomposition of a GEMM in CUTLASS.
    """
    # Grid level
    grid_m = (M + tb_m - 1) // tb_m
    grid_n = (N + tb_n - 1) // tb_n
    k_tiles = (K + tb_k - 1) // tb_k
    
    # Warps per thread block
    warps_m = tb_m // warp_m
    warps_n = tb_n // warp_n
    total_warps = warps_m * warps_n
    
    # MMA instructions per warp
    mmas_m = warp_m // mma_m
    mmas_n = warp_n // mma_n
    mmas_k = warp_k // mma_k
    
    print(f"CUTLASS GEMM Decomposition: ({M}×{K}) @ ({K}×{N})")
    print("=" * 70)
    
    print(f"\nLevel 0 — Grid:")
    print(f"  Thread blocks: {grid_m} × {grid_n} = {grid_m * grid_n}")
    print(f"  Each block computes: {tb_m} × {tb_n} tile of C")
    print(f"  K-dimension tiles: {k_tiles} iterations per block")
    
    print(f"\nLevel 1 — Thread Block ({tb_m}×{tb_n}×{tb_k}):")
    print(f"  Warps: {warps_m} × {warps_n} = {total_warps}")
    print(f"  Threads: {total_warps * 32} per block")
    print(f"  Shared memory per tile:")
    smem_a = tb_m * tb_k * 2  # FP16
    smem_b = tb_k * tb_n * 2
    print(f"    A tile: {tb_m}×{tb_k} × 2B = {smem_a/1024:.1f} KB")
    print(f"    B tile: {tb_k}×{tb_n} × 2B = {smem_b/1024:.1f} KB")
    print(f"    Total: {(smem_a + smem_b)/1024:.1f} KB")
    
    print(f"\nLevel 2 — Warp ({warp_m}×{warp_n}×{warp_k}):")
    print(f"  MMA instructions per warp per K-tile:")
    print(f"    {mmas_m} × {mmas_n} × {mmas_k} = {mmas_m * mmas_n * mmas_k} MMA ops")
    
    print(f"\nLevel 3 — MMA ({mma_m}×{mma_n}×{mma_k}):")
    print(f"  FLOPs per MMA: 2 × {mma_m} × {mma_n} × {mma_k} = {2*mma_m*mma_n*mma_k}")
    
    # Total FLOPs
    total_flops = 2 * M * N * K
    total_mma_ops = grid_m * grid_n * k_tiles * total_warps * mmas_m * mmas_n * mmas_k
    flops_per_mma = 2 * mma_m * mma_n * mma_k
    
    print(f"\nTotal FLOPs: {total_flops/1e9:.1f} GFLOPS")
    print(f"Total MMA ops: {total_mma_ops:,}")
    print(f"Verification: {total_mma_ops * flops_per_mma / 1e9:.1f} GFLOPS")

visualize_cutlass_decomposition()

## 3. CuTe: CUDA Tensor Expressions

### 3.1 What is CuTe?

CuTe is a C++ library within CUTLASS 3.x that provides a unified abstraction for tensor layouts and operations. It replaces the manual index arithmetic of CUTLASS 2.x.

### 3.2 Layout Abstraction

CuTe represents tensor layouts as `Layout<Shape, Stride>` pairs:

```cpp
// A 4×8 matrix in row-major order:
// Shape = (4, 8), Stride = (8, 1)
// Element at (i, j) is at offset: i * 8 + j * 1
auto layout = make_layout(make_shape(4, 8), make_stride(8, 1));

// Column-major: Stride = (1, 4)
auto col_layout = make_layout(make_shape(4, 8), make_stride(1, 4));

// Hierarchical (tiled) layout:
// A 128×128 matrix tiled into 32×32 blocks
auto tiled = make_layout(
    make_shape(make_shape(4, 32), make_shape(4, 32)),  // (4 blocks × 32, 4 blocks × 32)
    make_stride(make_stride(32*128, 1), make_stride(32, 128))  // Block-major strides
);
```

### 3.3 Key CuTe Operations

| Operation | Description |
|-----------|------------|
| `make_tensor(ptr, layout)` | Create a tensor view |
| `local_tile(tensor, tile_shape, coord)` | Extract a tile |
| `local_partition(tensor, thread_layout, thread_idx)` | Partition across threads |
| `copy(src, dst)` | Async copy between memory spaces |
| `gemm(mma, tA, tB, tC)` | Perform MMA on tensor fragments |

In [ ]:
# Simulate CuTe layout algebra in Python
import numpy as np

class Layout:
    """Simplified CuTe-style layout for educational purposes."""
    
    def __init__(self, shape, stride):
        self.shape = shape
        self.stride = stride
    
    def __repr__(self):
        return f"Layout(shape={self.shape}, stride={self.stride})"
    
    def offset(self, *coords):
        """Compute linear offset for given coordinates."""
        return sum(c * s for c, s in zip(coords, self.stride))
    
    def size(self):
        result = 1
        for s in self.shape:
            result *= s
        return result


def demonstrate_cute_layouts():
    print("CuTe Layout Examples")
    print("=" * 60)
    
    # Row-major 4×8 matrix
    row_major = Layout(shape=(4, 8), stride=(8, 1))
    print(f"\nRow-major {row_major.shape}: {row_major}")
    print(f"  Element (0,0) at offset {row_major.offset(0,0)}")
    print(f"  Element (1,3) at offset {row_major.offset(1,3)}")
    print(f"  Element (3,7) at offset {row_major.offset(3,7)}")
    
    # Column-major 4×8 matrix
    col_major = Layout(shape=(4, 8), stride=(1, 4))
    print(f"\nCol-major {col_major.shape}: {col_major}")
    print(f"  Element (0,0) at offset {col_major.offset(0,0)}")
    print(f"  Element (1,3) at offset {col_major.offset(1,3)}")
    print(f"  Element (3,7) at offset {col_major.offset(3,7)}")
    
    # Visualize access patterns for a warp (32 threads)
    print(f"\nWarp Access Pattern — Thread Layout for 16×16 tile:")
    print(f"  Each thread processes 8 elements (16×16 / 32 = 8)")
    
    # Layout: 4 rows of 8 threads, each thread handles 2×4 elements
    thread_layout = Layout(shape=(4, 8), stride=(8, 1))
    print(f"  Thread layout: {thread_layout}")
    
    for thread_id in [0, 1, 7, 8, 31]:
        row = thread_id // 8
        col = thread_id % 8
        print(f"  Thread {thread_id:2d}: row={row}, col={col}, "
              f"processes elements at rows [{row*4}:{row*4+4}], cols [{col*2}:{col*2+2}]")

demonstrate_cute_layouts()

## 4. Tensor Core Operations

### 4.1 WMMA (Warp Matrix Multiply-Accumulate)

WMMA is the older, more portable API for tensor cores:

```cpp
#include <mma.h>
using namespace nvcuda::wmma;

// Declare fragments (register-level tensor tiles)
fragment<matrix_a, 16, 16, 16, half, row_major> a_frag;
fragment<matrix_b, 16, 16, 16, half, col_major> b_frag;
fragment<accumulator, 16, 16, 16, float> c_frag;

// Initialize accumulator
fill_fragment(c_frag, 0.0f);

// Load fragments from shared memory
load_matrix_sync(a_frag, smem_a, lda);
load_matrix_sync(b_frag, smem_b, ldb);

// Matrix multiply-accumulate
mma_sync(c_frag, a_frag, b_frag, c_frag);

// Store result
store_matrix_sync(output, c_frag, ldc, mem_row_major);
```

### 4.2 MMA (PTX-level)

MMA is the lower-level PTX instruction used by CUTLASS:

```
mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32
    {d0, d1, d2, d3},    // Accumulator (FP32)
    {a0, a1, a2, a3},    // Matrix A fragment (FP16)
    {b0, b1},            // Matrix B fragment (FP16)
    {c0, c1, c2, c3};    // Previous accumulator
```

### 4.3 Tensor Core Sizes by Architecture

| Architecture | MMA Shape | Data Types |
|-------------|-----------|------------|
| Volta (V100) | 8×8×4 | FP16 |
| Turing (T4) | 8×8×4, 8×8×16 | FP16, INT8, INT4 |
| Ampere (A100) | 16×8×16, 16×8×32 | FP16, BF16, FP64, INT8, INT4 |
| Hopper (H100) | 64×64×16 (WGMMA) | FP16, BF16, FP8, INT8 |

In [ ]:
# Demonstrate tensor core throughput
import torch
import time

def benchmark_tensor_cores():
    """Measure tensor core vs CUDA core throughput."""
    if not torch.cuda.is_available():
        print("GPU required")
        return
    
    sizes = [1024, 2048, 4096, 8192]
    
    print(f"{'Size':>6} | {'FP32 (ms)':>10} {'TFLOPS':>8} | {'FP16 (ms)':>10} {'TFLOPS':>8} | {'TC Speedup':>10}")
    print("-" * 65)
    
    for N in sizes:
        flops = 2 * N * N * N  # GEMM FLOPs
        
        # FP32 (CUDA cores)
        A32 = torch.randn(N, N, device='cuda', dtype=torch.float32)
        B32 = torch.randn(N, N, device='cuda', dtype=torch.float32)
        
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            _ = torch.mm(A32, B32)
        torch.cuda.synchronize()
        fp32_time = (time.perf_counter() - start) / 10 * 1000
        fp32_tflops = flops / (fp32_time / 1000) / 1e12
        
        # FP16 (tensor cores)
        A16 = A32.half()
        B16 = B32.half()
        
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            _ = torch.mm(A16, B16)
        torch.cuda.synchronize()
        fp16_time = (time.perf_counter() - start) / 10 * 1000
        fp16_tflops = flops / (fp16_time / 1000) / 1e12
        
        speedup = fp32_time / fp16_time
        print(f"{N:>6} | {fp32_time:>10.3f} {fp32_tflops:>7.1f} | {fp16_time:>10.3f} {fp16_tflops:>7.1f} | {speedup:>9.2f}x")

benchmark_tensor_cores()

## 5. CUTLASS 3.x with CuTe

### 5.1 Major Changes from 2.x to 3.x

| Aspect | CUTLASS 2.x | CUTLASS 3.x |
|--------|------------|------------|
| Layout algebra | Manual index arithmetic | CuTe layout compositions |
| Thread mapping | Fixed thread-to-data mapping | Flexible via CuTe partitions |
| Memory ops | Manual shared memory management | `cute::copy` with automatic scheduling |
| MMA ops | WMMA wrappers | `cute::gemm` with MMA atoms |
| Pipeline | Manual staging | `cute::PipelineTmaAsync` |
| Hopper support | Limited | Full TMA, WGMMA, cluster support |

### 5.2 CUTLASS 3.x GEMM Structure

```cpp
// CUTLASS 3.x GEMM with CuTe (simplified)
template <class ProblemShape, class CtaTiler, class SmemLayout, class MmaAtom>
__global__ void gemm_kernel(ProblemShape shape, ...) {
    // 1. Create global tensors with CuTe
    auto gA = make_tensor(make_gmem_ptr(A), select<0,2>(shape), stride_A);
    auto gB = make_tensor(make_gmem_ptr(B), select<1,2>(shape), stride_B);
    auto gC = make_tensor(make_gmem_ptr(C), select<0,1>(shape), stride_C);
    
    // 2. Tile the problem
    auto cta_coord = make_coord(blockIdx.x, blockIdx.y, _);
    auto gA_tile = local_tile(gA, CtaTiler{}, cta_coord, Step<_1,X,_1>{});
    auto gB_tile = local_tile(gB, CtaTiler{}, cta_coord, Step<X,_1,_1>{});
    
    // 3. Partition across threads
    auto tAgA = local_partition(gA_tile, thread_layout, threadIdx.x);
    auto tBgB = local_partition(gB_tile, thread_layout, threadIdx.x);
    
    // 4. Mainloop: iterate over K tiles
    auto fragment_C = make_fragment_like(tCsC);
    clear(fragment_C);
    
    for (int k = 0; k < K_tiles; k++) {
        copy(tAgA, tAsA);  // Global → Shared
        copy(tBgB, tBsB);
        cp_async_fence();
        cp_async_wait<0>();
        __syncthreads();
        
        gemm(mma, tCsA, tCsB, fragment_C);  // Tensor core MMA
    }
    
    // 5. Epilogue: store result
    copy(fragment_C, tCgC);
}
```

In [ ]:
# Simulate the CUTLASS tiling hierarchy
def simulate_cutlass_tiling(M, N, K, cta_m, cta_n, cta_k, warp_m, warp_n, mma_m, mma_n, mma_k):
    """
    Simulate the three-level tiling used in CUTLASS.
    """
    # CTA (thread block) level
    num_ctas_m = (M + cta_m - 1) // cta_m
    num_ctas_n = (N + cta_n - 1) // cta_n
    k_iters = (K + cta_k - 1) // cta_k
    
    # Warp level
    warps_m = cta_m // warp_m
    warps_n = cta_n // warp_n
    num_warps = warps_m * warps_n
    
    # MMA level
    mma_m_iters = warp_m // mma_m
    mma_n_iters = warp_n // mma_n
    mma_k_iters = cta_k // mma_k
    
    print(f"CUTLASS Tiling: ({M}×{K}) @ ({K}×{N})")
    print(f"CTA tile: {cta_m}×{cta_n}×{cta_k}")
    print(f"Warp tile: {warp_m}×{warp_n}")
    print(f"MMA: {mma_m}×{mma_n}×{mma_k}")
    print()
    
    print(f"Grid: {num_ctas_m}×{num_ctas_n} = {num_ctas_m * num_ctas_n} CTAs")
    print(f"Warps/CTA: {warps_m}×{warps_n} = {num_warps}")
    print(f"Threads/CTA: {num_warps * 32}")
    print(f"K iterations: {k_iters}")
    print(f"MMA ops/warp/K-iter: {mma_m_iters}×{mma_n_iters}×{mma_k_iters} = {mma_m_iters*mma_n_iters*mma_k_iters}")
    print()
    
    # Compute efficiency metrics
    total_flops = 2 * M * N * K
    useful_flops = total_flops
    total_mma_flops = (num_ctas_m * num_ctas_n * k_iters * 
                       num_warps * mma_m_iters * mma_n_iters * mma_k_iters * 
                       2 * mma_m * mma_n * mma_k)
    
    efficiency = useful_flops / total_mma_flops * 100
    print(f"Computational efficiency: {efficiency:.1f}%")
    print(f"  (Accounts for edge effects when dimensions aren't perfect multiples)")

# Typical CUTLASS config for A100
simulate_cutlass_tiling(
    M=4096, N=4096, K=4096,
    cta_m=128, cta_n=128, cta_k=32,
    warp_m=64, warp_n=64,
    mma_m=16, mma_n=8, mma_k=16  # Ampere MMA shape
)

## 6. GEMM Epilogue Customization

### 6.1 What is an Epilogue?

The epilogue is the post-GEMM processing applied to the output tile before writing to global memory. Common epilogues:

```
Standard: C = alpha * A @ B + beta * C
Bias:     C = A @ B + bias
ReLU:     C = max(0, A @ B + bias)
GELU:     C = GELU(A @ B + bias)
SiLU-Mul: C = SiLU(A @ B_gate) * (A @ B_up)   — For LLaMA FFN
Quant:    C = quantize(A @ B)                    — For output quantization
```

### 6.2 Why Fuse the Epilogue?

The GEMM output tile is already in registers. Applying the epilogue:
- **Avoids writing to HBM** then reading back for the next operation
- **Saves memory bandwidth** — the main bottleneck
- Can fuse 2-3 operations for free since the data is already in registers

In [ ]:
# Demonstrate epilogue fusion benefit
import torch
import time

if torch.cuda.is_available():
    M, N, K = 4096, 4096, 4096
    
    A = torch.randn(M, K, device='cuda', dtype=torch.float16)
    B = torch.randn(K, N, device='cuda', dtype=torch.float16)
    bias = torch.randn(N, device='cuda', dtype=torch.float16)
    
    # Unfused: GEMM + bias + ReLU (3 separate kernels)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(50):
        C = torch.mm(A, B)          # Kernel 1: GEMM
        C = C + bias                 # Kernel 2: Bias add
        C = torch.relu(C)            # Kernel 3: ReLU
    torch.cuda.synchronize()
    unfused_time = (time.perf_counter() - start) / 50 * 1000
    
    # "Fused": Using addmm + relu (PyTorch may fuse internally)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(50):
        C = torch.relu(torch.addmm(bias.unsqueeze(0).expand(M, -1), A, B))
    torch.cuda.synchronize()
    fused_time = (time.perf_counter() - start) / 50 * 1000
    
    print(f"GEMM + Bias + ReLU ({M}×{K} @ {K}×{N})")
    print(f"  Unfused (3 kernels): {unfused_time:.2f} ms")
    print(f"  Semi-fused:          {fused_time:.2f} ms")
    print(f"  Speedup: {unfused_time/fused_time:.2f}x")
    print()
    print("With CUTLASS epilogue fusion, GEMM + bias + activation")
    print("would be a single kernel launch with zero extra memory traffic.")

## 7. Integrating CUTLASS into Python

### 7.1 Methods

| Method | Complexity | Performance | Flexibility |
|--------|-----------|------------|------------|
| **PyTorch CUDA Extension** | Medium | High | Full |
| **cutlass_library.generator** | Low | High | Moderate |
| **CUTLASS Python bindings** | Low | High | Moderate |
| **torch.compile** | None | Auto | Limited |

### 7.2 Example: PyTorch CUDA Extension

In [ ]:
# Example CUTLASS integration (code shown for reference)

cutlass_example = '''
// cutlass_gemm.cu — CUTLASS GEMM with bias + SiLU epilogue
#include <cutlass/gemm/device/gemm_universal.h>
#include <cutlass/epilogue/thread/linear_combination_silu.h>

// Define the GEMM configuration
using GemmKernel = cutlass::gemm::device::GemmUniversal<
    cutlass::half_t,                    // Element A
    cutlass::layout::RowMajor,          // Layout A
    cutlass::half_t,                    // Element B  
    cutlass::layout::ColumnMajor,       // Layout B
    cutlass::half_t,                    // Element C
    cutlass::layout::RowMajor,          // Layout C
    float,                              // Accumulator
    cutlass::arch::OpClassTensorOp,     // Use tensor cores
    cutlass::arch::Sm80,                // Target: Ampere (A100)
    cutlass::gemm::GemmShape<128, 128, 32>,  // Threadblock tile
    cutlass::gemm::GemmShape<64, 64, 32>,    // Warp tile
    cutlass::gemm::GemmShape<16, 8, 16>,     // MMA instruction
    // Custom epilogue with SiLU activation:
    cutlass::epilogue::thread::LinearCombinationSiLU<
        cutlass::half_t, 8, float, float
    >,
    cutlass::gemm::threadblock::GemmIdentityThreadblockSwizzle<>,
    3  // Pipeline stages
>;

// Launch the GEMM
void run_gemm(torch::Tensor A, torch::Tensor B, torch::Tensor C) {
    GemmKernel gemm_op;
    GemmKernel::Arguments args(
        {M, N, K},
        {A.data_ptr<cutlass::half_t>(), K},
        {B.data_ptr<cutlass::half_t>(), K},
        {C.data_ptr<cutlass::half_t>(), N},
        {C.data_ptr<cutlass::half_t>(), N},
        {1.0f, 0.0f}
    );
    gemm_op(args);
}
'''

print("CUTLASS GEMM with custom epilogue (SiLU activation)")
print("=" * 60)
print()
print("This example shows how to configure CUTLASS with:")
print("  - FP16 inputs with FP32 accumulation")
print("  - Tensor core operations (OpClassTensorOp)")
print("  - 128×128×32 threadblock tile")
print("  - 64×64×32 warp tile")
print("  - 16×8×16 MMA instruction (Ampere)")
print("  - Custom SiLU epilogue")
print("  - 3-stage pipeline for async loads")
print()
print("The key advantage: SiLU activation is applied")
print("to the GEMM output while it's still in registers,")
print("eliminating one HBM read+write cycle.")

## 8. Performance Comparison: CUTLASS vs cuBLAS vs Triton

### 8.1 Typical Performance Ranges

| Kernel | A100 FP16 | A100 INT8 | H100 FP8 |
|--------|----------|----------|----------|
| cuBLAS | 100% (baseline) | ~100% | ~100% |
| CUTLASS | 95-100% | 95-100% | 95-100% |
| Triton | 80-95% | 70-90% | 80-90% |
| Naive CUDA | 10-30% | N/A | N/A |

### 8.2 When Each Wins

- **cuBLAS**: Standard GEMM, no customization needed
- **CUTLASS**: Custom epilogues, non-standard data types, research
- **Triton**: Rapid prototyping, non-GEMM kernels, mixed operations

In [ ]:
# Compare cuBLAS (via PyTorch) vs Triton matmul
import torch
import time

if torch.cuda.is_available():
    try:
        import triton
        import triton.language as tl
        has_triton = True
    except ImportError:
        has_triton = False
    
    sizes = [(1024, 1024, 1024), (4096, 4096, 4096), (4096, 11008, 4096)]
    
    print(f"{'M×N×K':>18} | {'cuBLAS (ms)':>11} {'TFLOPS':>8} | {'Triton (ms)':>11} {'TFLOPS':>8} | {'Ratio':>6}")
    print("-" * 75)
    
    for M, N, K in sizes:
        A = torch.randn(M, K, device='cuda', dtype=torch.float16)
        B = torch.randn(K, N, device='cuda', dtype=torch.float16)
        flops = 2 * M * N * K
        
        # cuBLAS
        for _ in range(5):  # warmup
            _ = torch.mm(A, B)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(50):
            _ = torch.mm(A, B)
        torch.cuda.synchronize()
        cublas_ms = (time.perf_counter() - start) / 50 * 1000
        cublas_tflops = flops / (cublas_ms / 1000) / 1e12
        
        triton_ms = cublas_ms * 1.1  # Estimate if Triton not available
        triton_tflops = cublas_tflops / 1.1
        
        ratio = cublas_ms / triton_ms
        print(f"{M}×{N}×{K:>5} | {cublas_ms:>11.3f} {cublas_tflops:>7.1f} | "
              f"{triton_ms:>11.3f} {triton_tflops:>7.1f} | {ratio:>5.2f}x")
    
    print("\nNote: Triton times are estimated. Actual Triton performance depends")
    print("on the specific kernel implementation and autotuning configuration.")

## 9. Summary

| Concept | Key Takeaway |
|---------|-------------|
| **CUTLASS** | Open-source GEMM library with full customization |
| **CuTe** | Layout algebra for clean, composable tensor operations |
| **Epilogue** | Fuse post-GEMM ops (bias, activation) for free |
| **Tensor cores** | Hardware MMA units, 10-20x faster than CUDA cores |
| **CUTLASS 3.x** | CuTe-based, Hopper support, async pipelines |
| **vs cuBLAS** | 95-100% perf with much more flexibility |
| **vs Triton** | Higher perf ceiling but C++ complexity |

## Exercises

### Exercise 1: Design a CUTLASS Configuration

For a model with hidden_size=8192 and FFN_size=28672 (LLaMA-70B):
1. Calculate the optimal threadblock tile size
2. Determine shared memory requirements
3. Check if it fits in A100's shared memory (164 KB max)

### Exercise 2: Epilogue Design

Design an epilogue that fuses: GEMM + bias + GELU + dropout. What are the register requirements?

### Exercise 3: CuTe Layout Exercise

Work out the CuTe layout for a 256×256 matrix stored in 16×16 tiles in column-major order within each tile.

In [ ]:
# Exercise 1: CUTLASS config design
def design_cutlass_config(M, N, K, smem_limit_kb=164):
    configs = [
        (256, 128, 32), (128, 256, 32), (128, 128, 32),
        (128, 128, 64), (64, 128, 64), (128, 64, 64),
        (64, 64, 64), (64, 64, 32),
    ]
    
    print(f"Problem: ({M}×{K}) @ ({K}×{N}), SMEM limit: {smem_limit_kb} KB")
    print(f"{'CTA Tile':>15} | {'SMEM (KB)':>10} | {'Fits':>5} | {'CTAs':>8} | {'Waves':>6}")
    print("-" * 55)
    
    num_sms = 108  # A100
    
    for cta_m, cta_n, cta_k in configs:
        # Shared memory: A tile + B tile, FP16 (2 bytes), double buffered
        smem_a = cta_m * cta_k * 2  # bytes
        smem_b = cta_k * cta_n * 2
        total_smem = (smem_a + smem_b) * 2  # Double buffer
        total_smem_kb = total_smem / 1024
        
        fits = total_smem_kb <= smem_limit_kb
        
        num_ctas = ((M + cta_m - 1) // cta_m) * ((N + cta_n - 1) // cta_n)
        waves = (num_ctas + num_sms - 1) // num_sms
        
        print(f"{cta_m}×{cta_n}×{cta_k:>3} | {total_smem_kb:>8.1f} | {'YES' if fits else 'NO ':>5} | {num_ctas:>8} | {waves:>6}")

design_cutlass_config(M=8192, N=28672, K=8192)